In [3]:
from pypdf import PdfReader
import pdfplumber
from pathlib import Path
import os

from pdf2image import convert_from_path
import pytesseract

from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph

import pandas as pd

In [4]:
FOOTER_ZONE = 60   # points from bottom of page to exclude

In [5]:
PATH = r"/home/lenovo/projects/nilachala-policy-assistant/data/raw/"

In [6]:
docx_path = PATH + "it_support_escalation_matrix.docx"

In [7]:
def extract_document(file_path, row):
    ext = Path(file_path).suffix.lower()

    if ext == ".pdf":
        raw_pages = extract_pdf(file_path)
    elif ext == ".docx":
        raw_pages = extract_docx(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    pages = []
    for page_num, text, used_ocr in raw_pages:
        pages.append({
            "text": text,
            "page": page_num,
            "used_ocr": used_ocr,
            "doc_id": row.name,
            "source_file": row["filename"],
            "title": row["title"],
            "department": row["department"],
            "version": str(row["version"]),
            "is_current": str(row["is_current"]).strip().upper() == "TRUE",
        })
    return pages
	
	
def table_to_markdown(table):
    if not table:
        return ""
    
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).replace("\n", " ").strip()

    md_str = ""

    for i, row in enumerate(table):
        row_str = "|"
        for j in range(len(row)):
            row_str += f" {clean_cell(row[j])} |"
        row_str += "\n"
        mid_str = ""
        if i == 0:
            mid_str = "|" + " --- |" * len(row) + "\n"
        md_str += (row_str + mid_str)

    return md_str
	
	
def extract_page(page):
    tables = page.find_tables()

    usable_bottom = page.height - FOOTER_ZONE      # <-- add

    parts = []
    cursor = 0

    for t in tables:
        top, bottom = t.bbox[1], t.bbox[3]
        strip = page.crop((0, cursor, page.width, top))
        parts.append(strip.extract_text() or "")
        parts.append(table_to_markdown(t.extract()))
        cursor = bottom

    # anything below the last table, stopping before the footer
    parts.append(page.crop((0, cursor, page.width, usable_bottom)).extract_text() or "")

    return "\n\n".join(p.strip() for p in parts if p.strip())


def ocr_page(pdf_path, page_number, dpi=300):
    images = convert_from_path(str(pdf_path), dpi=dpi,
                               first_page=page_number, last_page=page_number)
    img = images[0]
    # FOOTER_ZONE is in PDF points (72/inch); scale to pixels at this dpi
    footer_px = int(FOOTER_ZONE * dpi / 72)
    img = img.crop((0, 0, img.width, img.height - footer_px))
    return pytesseract.image_to_string(img)


def extract_pdf(path):
    with pdfplumber.open(path) as pdf:
        pages = []
        for i, page in enumerate(pdf.pages):
            text = extract_page(page)
            used_ocr = False
            if len(text) < 50:
                text = ocr_page(path, i + 1)
                used_ocr = True
            pages.append((i + 1, text, used_ocr))
    return pages

def extract_docx(path):
    doc = Document(str(path))
    parts = []

    for child in doc.element.body.iterchildren():
        if child.tag.endswith("}p"):
            text = Paragraph(child, doc).text.strip()
            if text:
                parts.append(text)
        elif child.tag.endswith("}tbl"):
            table = Table(child, doc)
            rows = [[cell.text for cell in row.cells] for row in table.rows]
            parts.append(table_to_markdown(rows))

    return [(1, "\n\n".join(parts), False)]

In [10]:
pages = extract_docx(docx_path)
print(pages[0][1])

IT Support and Escalation Matrix

Nilachala Textiles Pvt. Ltd., Bhubaneswar
Document ref: D08  |  Version 1.2  |  Owner: IT department

1. Purpose

This document defines the internal service levels of the IT department and the escalation path for unresolved incidents.

This document is for the internal use of the IT department. It is not circulated to other departments.

2. Incident Priority Definitions

Table 2.1 - Priority definitions and response targets

| Priority | Definition | Response time | Resolution target |
| --- | --- | --- | --- |
| P1 | Production line stopped or full network outage | 15 minutes | 4 hours |
| P2 | Business function degraded, workaround exists | 1 hour | 1 working day |
| P3 | Single user unable to work | 4 hours | 2 working days |
| P4 | Request or minor issue, no work impact | 1 working day | 5 working days |


Priority is assigned by the IT helpdesk on logging and may be revised by the IT Manager. The requester may request a review of the assigned prio

In [35]:
manifest = pd.read_csv("/home/lenovo/projects/nilachala-policy-assistant/data/corpus_manifest.csv").set_index("doc_id")
print(manifest.loc["D01"])

filename                      casual_earned_leave_policy_v3.pdf
title                            Casual and Earned Leave Policy
department                                                  ALL
format                                                      pdf
pages                                                         8
is_scanned                                                False
has_tables                                                 True
version                                                     3.0
is_current                                                 True
notes         Current leave policy. Entitlement table: 12 ca...
Name: D01, dtype: object


In [26]:
row = manifest.loc["D01"]
print(row["filename"])

casual_earned_leave_policy_v3.pdf


In [38]:
pages = extract_document(PATH + row["filename"], row)
print(len(pages))
print(pages[0]["doc_id"], pages[0]["is_current"], type(pages[0]["is_current"]))
print(pages[0]["text"][:200])

2
D01 True <class 'bool'>
Casual and Earned Leave Policy
Nilachala Textiles Pvt. Ltd., Bhubaneswar
Document ref: D01 | Version 3.0 | Owner: ALL department
1. Purpose and Scope
This policy defines the leave entitlement availabl


In [39]:
all_pages = []
for doc_id, row in manifest.iterrows():
    pages = extract_document(PATH + row["filename"], row)
    all_pages.extend(pages)
    ocr_pages = sum(1 for p in pages if p["used_ocr"])
    print(f"{doc_id}  {len(pages):3d} pages  {ocr_pages} via OCR  {row['title'][:40]}")

print(f"\nTotal pages: {len(all_pages)}")
print(f"Total chars: {sum(len(p['text']) for p in all_pages):,}")
print(f"Departments: {set(p['department'] for p in all_pages)}")
print(f"Superseded:  {[p['doc_id'] for p in all_pages if not p['is_current']]}")

D01    2 pages  0 via OCR  Casual and Earned Leave Policy
D02    1 pages  0 via OCR  Leave Policy
D03   11 pages  0 via OCR  Employee Handbook
D04    2 pages  2 via OCR  Factory Floor Safety Manual
D05    2 pages  2 via OCR  Fire and Evacuation Procedure
D06    1 pages  1 via OCR  Machine Operating Guidelines
D07    2 pages  0 via OCR  Laptop and IT Asset Policy
D08    1 pages  0 via OCR  IT Support and Escalation Matrix
D09    2 pages  0 via OCR  Travel and Expense Reimbursement Policy
D10    2 pages  0 via OCR  Salary Grade and Allowance Structure
D11    2 pages  0 via OCR  Provident Fund and Gratuity Guidelines
D12    1 pages  0 via OCR  Code of Conduct and Disciplinary Procedu

Total pages: 29
Total chars: 47,073
Departments: {'ALL', 'Production', 'IT', 'Finance'}
Superseded:  ['D02']


In [40]:
import re

text = all_pages[0]["text"]
parts = re.split(r"\n(?=\d+\.\s)", text)
print(len(parts))
for p in parts:
    print(repr(p[:60]), len(p))

5
'Casual and Earned Leave Policy\nNilachala Textiles Pvt. Ltd.,' 128
'1. Purpose and Scope\nThis policy defines the leave entitleme' 578
'2. Leave Year\nThe leave year runs from 1 January to 31 Decem' 251
'3. Leave Entitlement\nTable 3.1 - Annual leave entitlement by' 728
'4. Application Procedure\nAll leave must be applied for throu' 165


In [6]:
# Create a PdfReader object
reader = PdfReader(pdf_path)

# Loop through all pages and extract text
for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    print(f"--- Page {page_num} ---")
    print(text)
    print("\n")


--- Page 1 ---
Nilachala Textiles Pvt. Ltd. - Internal document. Not for circulation outside the organisation.
Page 1
 Travel and Expense Reimbursement Policy
Nilachala Textiles Pvt. Ltd., Bhubaneswar
Document ref: D09   |   Version 2.0   |   Owner: ALL department
1. Scope and Principle
This policy covers expenses incurred by employees while travelling on company business.
Employees are expected to exercise the same care in incurring expenses on company business as they
would in managing their own affairs. Expenses must be reasonable, necessary and supported by
evidence.
2. Prior Approval
All business travel requires prior written approval from the department head using Form FIN-12.
International travel additionally requires the approval of the Managing Director.
3. Travel Entitlement by Grade
Table 3.1 - Mode of travel entitlement
Grade
Rail
Air
Local transport
Grade 1 - 2
Sleeper class
Not permitted
Bus or shared auto
Grade 3 - 4
AC 3 tier
Not permitted
Auto rickshaw
Grade 5 - 6
AC 2

In [7]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    print(f"Tables found: {len(tables)}")
    for t in tables:
        for row in t:
            print(row)

Tables found: 2
['Grade', 'Rail', 'Air', 'Local transport']
['Grade 1 - 2', 'Sleeper class', 'Not permitted', 'Bus or shared auto']
['Grade 3 - 4', 'AC 3 tier', 'Not permitted', 'Auto rickshaw']
['Grade 5 - 6', 'AC 2 tier', 'Economy, over 500 km', 'Taxi']
['Grade 7 and above', 'AC 1 tier', 'Economy', 'Taxi']
['Grade', 'Metro cities', 'Other cities', 'Lodging ceiling']
['Grade 1 - 2', 'Rs. 600', 'Rs. 450', 'Rs. 1,500']
['Grade 3 - 4', 'Rs. 900', 'Rs. 700', 'Rs. 2,500']
['Grade 5 - 6', 'Rs. 1,400', 'Rs. 1,100', 'Rs. 4,000']
['Grade 7 and above', 'Rs. 2,000', 'Rs. 1,600', 'Rs. 6,000']


In [8]:
print(tables), print(len(tables)), print(tables[0]), print(len(tables[0])), print(tables[0][0]), print(len(tables[0][0]))

[[['Grade', 'Rail', 'Air', 'Local transport'], ['Grade 1 - 2', 'Sleeper class', 'Not permitted', 'Bus or shared auto'], ['Grade 3 - 4', 'AC 3 tier', 'Not permitted', 'Auto rickshaw'], ['Grade 5 - 6', 'AC 2 tier', 'Economy, over 500 km', 'Taxi'], ['Grade 7 and above', 'AC 1 tier', 'Economy', 'Taxi']], [['Grade', 'Metro cities', 'Other cities', 'Lodging ceiling'], ['Grade 1 - 2', 'Rs. 600', 'Rs. 450', 'Rs. 1,500'], ['Grade 3 - 4', 'Rs. 900', 'Rs. 700', 'Rs. 2,500'], ['Grade 5 - 6', 'Rs. 1,400', 'Rs. 1,100', 'Rs. 4,000'], ['Grade 7 and above', 'Rs. 2,000', 'Rs. 1,600', 'Rs. 6,000']]]
2
[['Grade', 'Rail', 'Air', 'Local transport'], ['Grade 1 - 2', 'Sleeper class', 'Not permitted', 'Bus or shared auto'], ['Grade 3 - 4', 'AC 3 tier', 'Not permitted', 'Auto rickshaw'], ['Grade 5 - 6', 'AC 2 tier', 'Economy, over 500 km', 'Taxi'], ['Grade 7 and above', 'AC 1 tier', 'Economy', 'Taxi']]
5
['Grade', 'Rail', 'Air', 'Local transport']
4


(None, None, None, None, None, None)

In [10]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    print(f"Tables found: {len(tables)}")
    for t in tables:
        # for row in t:
        #     print(row)
        print(table_to_markdown(t))
        print()

Tables found: 2
| Grade | Rail | Air | Local transport |
| --- | --- | --- | --- |
| Grade 1 - 2 | Sleeper class | Not permitted | Bus or shared auto |
| Grade 3 - 4 | AC 3 tier | Not permitted | Auto rickshaw |
| Grade 5 - 6 | AC 2 tier | Economy, over 500 km | Taxi |
| Grade 7 and above | AC 1 tier | Economy | Taxi |


| Grade | Metro cities | Other cities | Lodging ceiling |
| --- | --- | --- | --- |
| Grade 1 - 2 | Rs. 600 | Rs. 450 | Rs. 1,500 |
| Grade 3 - 4 | Rs. 900 | Rs. 700 | Rs. 2,500 |
| Grade 5 - 6 | Rs. 1,400 | Rs. 1,100 | Rs. 4,000 |
| Grade 7 and above | Rs. 2,000 | Rs. 1,600 | Rs. 6,000 |




In [11]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    for t in page.find_tables():
        print(t.bbox)          # (x0, top, x1, bottom)

(68.3622, 395.37800000000004, 526.9134, 495.37800000000004)
(68.3622, 569.0552, 526.9134, 669.0552)


In [26]:
for docx_file in sorted(Path(PATH).glob("*.docx")):
    pages = extract_docx(docx_file)
    total_chars = sum(len(t) for _, t, _ in pages)
    print(f"{docx_file.name:42s} {len(pages):3d} pages {total_chars:7d} chars")

code_of_conduct.docx                         1 pages    3734 chars
it_support_escalation_matrix.docx            1 pages    1836 chars


In [25]:
for pdf_file in sorted(Path(PATH).glob("*.pdf")):
    pages = extract_pdf(pdf_file)
    total_chars = sum(len(t) for _, t, _ in pages)
    print(f"{pdf_file.name:42s} {len(pages):3d} pages {total_chars:7d} chars")

casual_earned_leave_policy_v3.pdf            2 pages    3324 chars
employee_handbook.pdf                       11 pages   20475 chars
factory_floor_safety_manual.pdf              2 pages    2978 chars
fire_evacuation_procedure.pdf                2 pages    2465 chars
laptop_it_asset_policy.pdf                   2 pages    2293 chars
leave_policy_2019.pdf                        1 pages    1157 chars
machine_operating_guidelines.pdf             1 pages    1772 chars
provident_fund_gratuity_guidelines.pdf       2 pages    2572 chars
salary_grade_structure.pdf                   2 pages    2145 chars
travel_expense_reimbursement.pdf             2 pages    2322 chars


In [27]:
path = "/home/lenovo/projects/nilachala-policy-assistant/data/raw/"
file_names_list = [file_name for file_name in os.listdir(path) if file_name.lower().endswith((".pdf",".docx"))]
print(file_names_list), print(len(file_names_list))

['machine_operating_guidelines.pdf', 'code_of_conduct.docx', 'leave_policy_2019.pdf', 'employee_handbook.pdf', 'fire_evacuation_procedure.pdf', 'travel_expense_reimbursement.pdf', 'factory_floor_safety_manual.pdf', 'it_support_escalation_matrix.docx', 'casual_earned_leave_policy_v3.pdf', 'salary_grade_structure.pdf', 'laptop_it_asset_policy.pdf', 'provident_fund_gratuity_guidelines.pdf']
12


(None, None)

In [21]:
# def clean_cell(cell):
#     if cell == None:
#         return " "
#     return cell

# table = tables[0]

# md_str = ""

# for i, row in enumerate(table):
#     row_str = "|"
#     for j in range(len(row)):
#         row_str += f" {clean_cell(row[j])} |"
#     row_str += "\n"
#     mid_str = ""
#     if i == 0:
#         mid_str = "|" + " --- |" * len(row) + "\n"
#     md_str += (row_str + mid_str)

# print(md_str)